# Module 5: Fair Value Model Fundamentals

## Learning Objectives

By the end of this module, you will:
1. Understand the theory behind different fair value models
2. Implement inventory-based models using storage theory
3. Build cost-of-carry models and estimate convenience yield
4. Create supply-demand balance models
5. Combine multiple models using ensemble approaches
6. Evaluate and compare model performance

## What is Fair Value?

**Fair value** is an estimate of what a commodity "should" be worth based on fundamental economic factors, as opposed to its current market price.

**Key insight**: When market price deviates from fair value, it may signal:
- Trading opportunities (reversion to fair value)
- Regime changes (fair value model breakdown)
- Structural shifts (need to update model)

Fair value is **not** a prediction of future prices, but rather a fundamental anchor.

In [ ]:
# Setup and imports
import sys
sys.path.append('../..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta
from scipy import stats, optimize
from sklearn.linear_model import LinearRegression, Ridge, ElasticNet
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import warnings
warnings.filterwarnings('ignore')

# Fair Value Toolkit imports
from src.fair_value_toolkit import (
    PointInTimeDataFrame,
    FairValueModel,
    InventoryFairValueModel,
    CostOfCarryModel,
    SupplyDemandModel,
    EnsembleFairValueModel,
    WalkForwardValidator,
)

from data.data_fetchers import (
    create_sample_dataset,
    prepare_training_data,
)

# Plotting configuration
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")
plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['font.size'] = 10

print("Environment ready!")

## Data Preparation

Let's load our cleaned commodity data for model building.

In [ ]:
# Create comprehensive dataset
data = create_sample_dataset(
    commodity='crude_oil',
    start_date='2018-01-01',
    end_date='2023-12-31',
    include_revisions=False  # Use final data for cleaner examples
)

# Convert to point-in-time structure
pit_data = PointInTimeDataFrame(data)

print(f"Dataset shape: {data.shape}")
print(f"Columns: {list(data.columns)}")
print(f"Date range: {data.index.min().date()} to {data.index.max().date()}")
data.head()

In [ ]:
# Visualize key variables
fig, axes = plt.subplots(3, 1, figsize=(14, 12), sharex=True)

# Price
data['price'].plot(ax=axes[0], linewidth=1.5, color='blue')
axes[0].set_ylabel('Price ($/barrel)')
axes[0].set_title('WTI Crude Oil Price')
axes[0].grid(True, alpha=0.3)

# Inventory
data['inventory'].plot(ax=axes[1], linewidth=1.5, color='brown')
axes[1].set_ylabel('Inventory (million barrels)')
axes[1].set_title('U.S. Crude Oil Inventory')
axes[1].grid(True, alpha=0.3)

# Production
data['production'].plot(ax=axes[2], linewidth=1.5, color='green')
axes[2].set_ylabel('Production (million barrels/day)')
axes[2].set_xlabel('Date')
axes[2].set_title('U.S. Crude Oil Production')
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Part 1: Inventory-Based Models

### 1.1 Theory of Storage (Kaldor, Working)

The **theory of storage** posits that commodity prices are related to inventory levels:

**Key relationships**:
- High inventory → Low prices (abundant supply)
- Low inventory → High prices (scarcity)
- Inventory levels reflect supply/demand balance

**Working's curve**: Non-linear relationship between inventory and price
- At very low inventory: Price becomes highly sensitive ("scarcity premium")
- At high inventory: Price less sensitive ("abundance discount")

### 1.2 Simple Inventory Model

In [ ]:
# Create normalized inventory (z-score)
data['inventory_zscore'] = (data['inventory'] - data['inventory'].expanding(min_periods=30).mean()) / \
                           data['inventory'].expanding(min_periods=30).std()

# Examine relationship
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Scatter plot
axes[0].scatter(data['inventory'], data['price'], alpha=0.5, s=20)
axes[0].set_xlabel('Inventory (million barrels)')
axes[0].set_ylabel('Price ($/barrel)')
axes[0].set_title('Price vs Inventory (Raw)')
axes[0].grid(True, alpha=0.3)

# Price vs normalized inventory
axes[1].scatter(data['inventory_zscore'], data['price'], alpha=0.5, s=20)
axes[1].set_xlabel('Inventory Z-Score')
axes[1].set_ylabel('Price ($/barrel)')
axes[1].set_title('Price vs Normalized Inventory')
axes[1].axvline(x=0, color='red', linestyle='--', alpha=0.5)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Calculate correlation
corr = data[['price', 'inventory']].corr().iloc[0, 1]
print(f"\nPrice-Inventory Correlation: {corr:.3f}")

### 1.3 InventoryFairValueModel Implementation

In [ ]:
# Split data into train and test
train_end = '2022-12-31'
train_data = data.loc[:train_end].copy()
test_data = data.loc[train_end:].copy()

print(f"Training period: {train_data.index.min().date()} to {train_data.index.max().date()}")
print(f"Test period: {test_data.index.min().date()} to {test_data.index.max().date()}")
print(f"Train samples: {len(train_data)}, Test samples: {len(test_data)}")

In [ ]:
# Initialize and fit inventory model
inventory_model = InventoryFairValueModel(
    name='inventory_model',
    functional_form='power',  # Non-linear relationship
    include_seasonal=True      # Account for seasonal patterns
)

# Prepare features and target
X_train = train_data[['inventory', 'production']]
y_train = train_data['price']

# Fit model
inventory_model.fit(X_train, y_train, as_of_date=pd.Timestamp(train_end))

print("\nModel fitted successfully!")
print(f"\nModel parameters:")
print(inventory_model._model_params)

In [ ]:
# Generate fair value estimates
X_test = test_data[['inventory', 'production']]
fair_values = inventory_model.predict(X_test, type='fair_value')

# Visualize results
fig, axes = plt.subplots(2, 1, figsize=(14, 10), sharex=True)

# Fair value vs actual price
test_data['price'].plot(ax=axes[0], linewidth=2, label='Actual Price', color='blue')
fair_values['fair_value'].plot(ax=axes[0], linewidth=2, label='Fair Value', color='green', linestyle='--')
axes[0].fill_between(
    fair_values.index,
    fair_values['lower_bound'],
    fair_values['upper_bound'],
    alpha=0.2,
    color='green',
    label='95% Confidence Interval'
)
axes[0].set_ylabel('Price ($/barrel)')
axes[0].set_title('Inventory Fair Value Model - Test Period')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Mispricing (deviation from fair value)
mispricing = test_data['price'] - fair_values['fair_value']
mispricing.plot(ax=axes[1], linewidth=1.5, color='red')
axes[1].axhline(y=0, color='black', linestyle='--', linewidth=2)
axes[1].fill_between(
    mispricing.index,
    mispricing,
    0,
    where=(mispricing > 0),
    alpha=0.3,
    color='green',
    label='Overvalued'
)
axes[1].fill_between(
    mispricing.index,
    mispricing,
    0,
    where=(mispricing < 0),
    alpha=0.3,
    color='red',
    label='Undervalued'
)
axes[1].set_ylabel('Mispricing ($/barrel)')
axes[1].set_xlabel('Date')
axes[1].set_title('Price Deviation from Fair Value')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Evaluate model performance
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

actual = test_data['price'].values
predicted = fair_values['fair_value'].values

mse = mean_squared_error(actual, predicted)
rmse = np.sqrt(mse)
mae = mean_absolute_error(actual, predicted)
r2 = r2_score(actual, predicted)

print("Model Performance Metrics:\n")
print(f"RMSE: ${rmse:.2f}/barrel")
print(f"MAE: ${mae:.2f}/barrel")
print(f"R²: {r2:.3f}")
print(f"\nMean absolute mispricing: ${np.abs(mispricing).mean():.2f}/barrel")
print(f"Max mispricing: ${mispricing.max():.2f}/barrel (overvalued)")
print(f"Min mispricing: ${mispricing.min():.2f}/barrel (undervalued)")

## Part 2: Cost-of-Carry Models

### 2.1 Theory of Storage Costs

The **cost-of-carry** model relates spot and futures prices:

$$F = S \cdot e^{(r + u - y) \cdot t}$$

Where:
- $F$ = Futures price
- $S$ = Spot price
- $r$ = Risk-free rate
- $u$ = Storage cost
- $y$ = Convenience yield
- $t$ = Time to maturity

**Convenience yield** ($y$) is the benefit of holding physical inventory:
- Ability to meet unexpected demand
- Avoid production interruptions
- Capture temporary price spikes

### 2.2 Estimating Convenience Yield

In [ ]:
# Calculate implied convenience yield from futures-spot spread
# Simplified: assume storage costs and interest rate are small

# Create synthetic futures price (spot + premium/discount)
data['futures_1m'] = data['price'] * (1 + np.random.randn(len(data)) * 0.02)

# Calculate convenience yield (annualized)
time_to_maturity = 1/12  # 1 month in years
risk_free_rate = 0.03    # 3% annually
storage_cost = 0.01      # 1% annually

# Implied convenience yield: y = r + u - ln(F/S) / t
data['convenience_yield'] = risk_free_rate + storage_cost - \
                            np.log(data['futures_1m'] / data['price']) / time_to_maturity

# Visualize
fig, axes = plt.subplots(2, 1, figsize=(14, 10), sharex=True)

# Futures premium
futures_premium = (data['futures_1m'] - data['price']) / data['price'] * 100
futures_premium.plot(ax=axes[0], linewidth=1.5)
axes[0].axhline(y=0, color='black', linestyle='--', linewidth=2)
axes[0].fill_between(
    futures_premium.index,
    futures_premium,
    0,
    where=(futures_premium > 0),
    alpha=0.3,
    color='green',
    label='Contango'
)
axes[0].fill_between(
    futures_premium.index,
    futures_premium,
    0,
    where=(futures_premium < 0),
    alpha=0.3,
    color='red',
    label='Backwardation'
)
axes[0].set_ylabel('Futures Premium (%)')
axes[0].set_title('Futures-Spot Spread (Market Structure)')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Convenience yield
data['convenience_yield'].plot(ax=axes[1], linewidth=1.5, color='purple')
axes[1].axhline(y=risk_free_rate + storage_cost, color='orange', linestyle='--', 
                label='r + u (breakeven)', linewidth=2)
axes[1].set_ylabel('Convenience Yield (annualized)')
axes[1].set_xlabel('Date')
axes[1].set_title('Implied Convenience Yield')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

### 2.3 CostOfCarryModel Implementation

In [ ]:
# Initialize cost-of-carry model
carry_model = CostOfCarryModel(
    name='cost_of_carry',
    storage_cost=0.01,  # 1% annually
    risk_free_rate=0.03  # 3% annually
)

# Prepare features (inventory level affects convenience yield)
X_train_carry = train_data[['inventory', 'production', 'demand']]
y_train_carry = train_data['price']

# Fit model
carry_model.fit(X_train_carry, y_train_carry, as_of_date=pd.Timestamp(train_end))

print("Cost-of-carry model fitted!")
print(f"\nModel parameters:")
print(carry_model._model_params)

In [ ]:
# Generate predictions
X_test_carry = test_data[['inventory', 'production', 'demand']]
carry_fair_values = carry_model.predict(X_test_carry, type='fair_value')

# Compare with inventory model
fig, ax = plt.subplots(figsize=(14, 6))

test_data['price'].plot(ax=ax, linewidth=2, label='Actual Price', color='blue')
fair_values['fair_value'].plot(ax=ax, linewidth=2, label='Inventory Model', color='green', linestyle='--')
carry_fair_values['fair_value'].plot(ax=ax, linewidth=2, label='Cost-of-Carry Model', color='orange', linestyle=':')

ax.set_ylabel('Price ($/barrel)')
ax.set_xlabel('Date')
ax.set_title('Model Comparison: Inventory vs Cost-of-Carry')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Compare performance
carry_rmse = np.sqrt(mean_squared_error(actual, carry_fair_values['fair_value'].values))
carry_r2 = r2_score(actual, carry_fair_values['fair_value'].values)

print("\nModel Comparison:\n")
print(f"Inventory Model - RMSE: ${rmse:.2f}, R²: {r2:.3f}")
print(f"Cost-of-Carry Model - RMSE: ${carry_rmse:.2f}, R²: {carry_r2:.3f}")

## Part 3: Supply-Demand Balance Models

### 3.1 Theory of Supply and Demand

Classic economic theory: price equilibrates supply and demand.

For commodities:
- **Supply factors**: Production, imports, inventory releases
- **Demand factors**: Consumption, exports, refinery utilization
- **Balance**: Inventory changes reflect supply-demand imbalance

**Model specification**:
$$P = f(Production, Demand, Inventory, Seasonality, Economic\_Indicators)$$

### 3.2 Feature Selection

In [ ]:
# Calculate derived features
data['supply_demand_balance'] = data['production'] - data['demand']
data['inventory_change'] = data['inventory'].diff()
data['production_growth'] = data['production'].pct_change()
data['demand_growth'] = data['demand'].pct_change()

# Examine correlations with price
feature_cols = [
    'production', 'demand', 'inventory', 'supply_demand_balance',
    'inventory_change', 'production_growth', 'demand_growth'
]

correlations = data[feature_cols + ['price']].corr()['price'].drop('price').sort_values(ascending=False)

print("Feature Correlations with Price:\n")
print(correlations)

# Visualize
fig, ax = plt.subplots(figsize=(10, 6))
correlations.plot(kind='barh', ax=ax, color=['green' if x > 0 else 'red' for x in correlations])
ax.set_xlabel('Correlation with Price')
ax.set_title('Supply-Demand Feature Correlations')
ax.axvline(x=0, color='black', linestyle='--', linewidth=2)
ax.grid(True, alpha=0.3, axis='x')
plt.tight_layout()
plt.show()

### 3.3 SupplyDemandModel Implementation

In [ ]:
# Initialize supply-demand model
sd_model = SupplyDemandModel(
    name='supply_demand',
    regularization='ridge',  # Ridge regression to handle multicollinearity
    alpha=1.0                # Regularization strength
)

# Prepare features
feature_cols_selected = [
    'production', 'demand', 'inventory', 'supply_demand_balance',
    'inventory_change'
]

X_train_sd = train_data[feature_cols_selected].dropna()
y_train_sd = train_data.loc[X_train_sd.index, 'price']

# Fit model
sd_model.fit(X_train_sd, y_train_sd, as_of_date=pd.Timestamp(train_end))

print("Supply-Demand model fitted!")
print(f"\nFeature coefficients:")
coefs = pd.DataFrame({
    'feature': feature_cols_selected,
    'coefficient': sd_model._model_params['coefficients']
}).sort_values('coefficient', ascending=False)
print(coefs.to_string(index=False))

In [ ]:
# Visualize coefficient importance
fig, ax = plt.subplots(figsize=(10, 6))

colors = ['green' if x > 0 else 'red' for x in coefs['coefficient']]
coefs.plot(x='feature', y='coefficient', kind='barh', ax=ax, color=colors, legend=False)
ax.set_xlabel('Coefficient')
ax.set_ylabel('Feature')
ax.set_title('Supply-Demand Model Feature Importance')
ax.axvline(x=0, color='black', linestyle='--', linewidth=2)
ax.grid(True, alpha=0.3, axis='x')

plt.tight_layout()
plt.show()

In [ ]:
# Generate predictions
X_test_sd = test_data[feature_cols_selected].dropna()
sd_fair_values = sd_model.predict(X_test_sd, type='fair_value')

# Compare all three models
fig, ax = plt.subplots(figsize=(14, 6))

test_data.loc[sd_fair_values.index, 'price'].plot(
    ax=ax, linewidth=2, label='Actual Price', color='blue'
)
fair_values.loc[sd_fair_values.index, 'fair_value'].plot(
    ax=ax, linewidth=2, label='Inventory Model', color='green', linestyle='--', alpha=0.7
)
carry_fair_values.loc[sd_fair_values.index, 'fair_value'].plot(
    ax=ax, linewidth=2, label='Cost-of-Carry Model', color='orange', linestyle=':', alpha=0.7
)
sd_fair_values['fair_value'].plot(
    ax=ax, linewidth=2, label='Supply-Demand Model', color='purple', linestyle='-.', alpha=0.7
)

ax.set_ylabel('Price ($/barrel)')
ax.set_xlabel('Date')
ax.set_title('Fair Value Model Comparison')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Compare performance
sd_actual = test_data.loc[sd_fair_values.index, 'price'].values
sd_predicted = sd_fair_values['fair_value'].values
sd_rmse = np.sqrt(mean_squared_error(sd_actual, sd_predicted))
sd_r2 = r2_score(sd_actual, sd_predicted)

print("\nModel Performance Summary:\n")
print(f"Inventory Model      - RMSE: ${rmse:.2f}, R²: {r2:.3f}")
print(f"Cost-of-Carry Model  - RMSE: ${carry_rmse:.2f}, R²: {carry_r2:.3f}")
print(f"Supply-Demand Model  - RMSE: ${sd_rmse:.2f}, R²: {sd_r2:.3f}")

## Part 4: Ensemble Approaches

### 4.1 Why Ensemble Models?

Different models capture different aspects of fair value:
- **Inventory model**: Storage dynamics
- **Cost-of-carry model**: Market structure
- **Supply-demand model**: Fundamental balance

**Ensemble benefits**:
- Diversification across model risk
- More robust to regime changes
- Better out-of-sample performance

### 4.2 Simple Average Ensemble

In [ ]:
# Create simple average ensemble
# Align all predictions to common index
common_index = sd_fair_values.index

ensemble_simple = pd.DataFrame({
    'inventory': fair_values.loc[common_index, 'fair_value'],
    'carry': carry_fair_values.loc[common_index, 'fair_value'],
    'supply_demand': sd_fair_values['fair_value']
})

ensemble_simple['average'] = ensemble_simple.mean(axis=1)

# Visualize
fig, axes = plt.subplots(2, 1, figsize=(14, 10), sharex=True)

# Individual models + ensemble
test_data.loc[common_index, 'price'].plot(ax=axes[0], linewidth=2.5, label='Actual Price', color='blue')
ensemble_simple['inventory'].plot(ax=axes[0], linewidth=1, label='Inventory', alpha=0.5)
ensemble_simple['carry'].plot(ax=axes[0], linewidth=1, label='Cost-of-Carry', alpha=0.5)
ensemble_simple['supply_demand'].plot(ax=axes[0], linewidth=1, label='Supply-Demand', alpha=0.5)
ensemble_simple['average'].plot(ax=axes[0], linewidth=2.5, label='Ensemble (Avg)', 
                                color='red', linestyle='--')
axes[0].set_ylabel('Price ($/barrel)')
axes[0].set_title('Ensemble Fair Value Model')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Model dispersion (uncertainty)
ensemble_std = ensemble_simple[['inventory', 'carry', 'supply_demand']].std(axis=1)
ensemble_std.plot(ax=axes[1], linewidth=1.5, color='purple')
axes[1].set_ylabel('Model Dispersion ($/barrel)')
axes[1].set_xlabel('Date')
axes[1].set_title('Model Disagreement (Ensemble Uncertainty)')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Evaluate ensemble
ensemble_rmse = np.sqrt(mean_squared_error(sd_actual, ensemble_simple['average'].values))
ensemble_r2 = r2_score(sd_actual, ensemble_simple['average'].values)

print("\nEnsemble Performance:\n")
print(f"Simple Average - RMSE: ${ensemble_rmse:.2f}, R²: {ensemble_r2:.3f}")

### 4.3 Weighted Ensemble (EnsembleFairValueModel)

In [ ]:
# Initialize ensemble model with optimized weights
ensemble_model = EnsembleFairValueModel(
    name='weighted_ensemble',
    models={
        'inventory': inventory_model,
        'carry': carry_model,
        'supply_demand': sd_model
    },
    weight_optimization='inverse_error'  # Weight by inverse RMSE
)

# Fit ensemble (learns optimal weights)
ensemble_model.fit(
    X_train_sd,  # Use supply-demand features (most comprehensive)
    y_train_sd,
    as_of_date=pd.Timestamp(train_end)
)

print("Weighted ensemble fitted!")
print(f"\nOptimized weights:")
for model_name, weight in ensemble_model._model_params['weights'].items():
    print(f"  {model_name}: {weight:.3f}")

In [ ]:
# Generate ensemble predictions
ensemble_predictions = ensemble_model.predict(X_test_sd, type='fair_value')

# Compare simple vs weighted ensemble
fig, ax = plt.subplots(figsize=(14, 6))

test_data.loc[common_index, 'price'].plot(ax=ax, linewidth=2.5, label='Actual Price', color='blue')
ensemble_simple['average'].plot(ax=ax, linewidth=2, label='Simple Average', 
                                color='orange', linestyle='--', alpha=0.7)
ensemble_predictions['fair_value'].plot(ax=ax, linewidth=2, label='Weighted Ensemble', 
                                       color='red', linestyle='--')

ax.set_ylabel('Price ($/barrel)')
ax.set_xlabel('Date')
ax.set_title('Simple vs Weighted Ensemble')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Final comparison
weighted_rmse = np.sqrt(mean_squared_error(sd_actual, ensemble_predictions['fair_value'].values))
weighted_r2 = r2_score(sd_actual, ensemble_predictions['fair_value'].values)

print("\n" + "="*60)
print("FINAL MODEL PERFORMANCE COMPARISON")
print("="*60)
print(f"Inventory Model         - RMSE: ${rmse:.2f}, R²: {r2:.3f}")
print(f"Cost-of-Carry Model     - RMSE: ${carry_rmse:.2f}, R²: {carry_r2:.3f}")
print(f"Supply-Demand Model     - RMSE: ${sd_rmse:.2f}, R²: {sd_r2:.3f}")
print(f"Simple Average Ensemble - RMSE: ${ensemble_rmse:.2f}, R²: {ensemble_r2:.3f}")
print(f"Weighted Ensemble       - RMSE: ${weighted_rmse:.2f}, R²: {weighted_r2:.3f}")
print("="*60)

### 4.4 Model Diagnostics

In [ ]:
# Residual analysis for weighted ensemble
residuals = sd_actual - ensemble_predictions['fair_value'].values

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 1. Residuals over time
pd.Series(residuals, index=ensemble_predictions.index).plot(ax=axes[0, 0], linewidth=1.5)
axes[0, 0].axhline(y=0, color='red', linestyle='--', linewidth=2)
axes[0, 0].set_ylabel('Residual ($/barrel)')
axes[0, 0].set_title('Residuals Over Time')
axes[0, 0].grid(True, alpha=0.3)

# 2. Histogram of residuals
axes[0, 1].hist(residuals, bins=30, edgecolor='black', alpha=0.7)
axes[0, 1].axvline(x=0, color='red', linestyle='--', linewidth=2)
axes[0, 1].set_xlabel('Residual ($/barrel)')
axes[0, 1].set_ylabel('Frequency')
axes[0, 1].set_title('Residual Distribution')
axes[0, 1].grid(True, alpha=0.3)

# 3. Q-Q plot
stats.probplot(residuals, dist="norm", plot=axes[1, 0])
axes[1, 0].set_title('Q-Q Plot (Normality Test)')
axes[1, 0].grid(True, alpha=0.3)

# 4. Residuals vs fitted
axes[1, 1].scatter(ensemble_predictions['fair_value'].values, residuals, alpha=0.5, s=20)
axes[1, 1].axhline(y=0, color='red', linestyle='--', linewidth=2)
axes[1, 1].set_xlabel('Fitted Value ($/barrel)')
axes[1, 1].set_ylabel('Residual ($/barrel)')
axes[1, 1].set_title('Residuals vs Fitted Values')
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Statistical tests
print("\nResidual Statistics:\n")
print(f"Mean: ${residuals.mean():.2f} (should be ~0)")
print(f"Std Dev: ${residuals.std():.2f}")
print(f"Skewness: {stats.skew(residuals):.3f} (0 = symmetric)")
print(f"Kurtosis: {stats.kurtosis(residuals):.3f} (0 = normal)")

# Ljung-Box test for autocorrelation
from statsmodels.stats.diagnostic import acorr_ljungbox
lb_test = acorr_ljungbox(residuals, lags=[10], return_df=True)
print(f"\nLjung-Box test (autocorrelation):")
print(f"p-value: {lb_test['lb_pvalue'].values[0]:.3f} (>0.05 = no autocorrelation)")

## Summary and Key Takeaways

### What We've Learned

1. **Fair Value Concepts**:
   - Fair value estimates fundamental price based on economics
   - Deviations signal potential trading opportunities
   - Different models capture different aspects of value

2. **Model Types**:
   - **Inventory models**: Simple, theory-driven, storage dynamics
   - **Cost-of-carry**: Market structure, convenience yield
   - **Supply-demand**: Comprehensive fundamentals
   - **Ensembles**: Combine strengths, reduce model risk

3. **Implementation**:
   - Use point-in-time data for training
   - Validate on out-of-sample test data
   - Check residuals for model assumptions
   - Consider ensemble approaches for robustness

4. **Model Selection**:
   - No single "best" model for all commodities/periods
   - Ensemble often provides best risk-adjusted performance
   - Model weights should adapt to changing market conditions

### Critical Considerations

1. **Data requirements vary by model**:
   - Inventory: Price + inventory levels
   - Cost-of-carry: Spot + futures + interest rates
   - Supply-demand: Production, consumption, inventory, economics

2. **Temporal discipline is crucial**:
   - Only use data available at prediction time
   - Account for publication delays
   - Use expanding windows for statistics

3. **Model monitoring**:
   - Track out-of-sample performance
   - Watch for regime changes
   - Refit periodically with new data

### Next Steps

In Module 6, we'll learn how to:
- Validate models using walk-forward analysis
- Detect and prevent data leakage
- Measure model decay over time
- Implement robust backtesting frameworks

## Exercises

1. **Custom Inventory Model**: Implement a non-linear inventory model using polynomial features or splines.

2. **Dynamic Weights**: Create an ensemble where model weights adapt based on recent performance (e.g., exponentially weighted moving average of errors).

3. **Regime-Dependent Models**: Build separate models for high/low volatility regimes and combine them.

4. **Feature Engineering**: Create additional features for the supply-demand model:
   - Seasonal indicators
   - Economic indicators (GDP, unemployment)
   - Geopolitical risk indices

5. **Model Confidence**: Implement a method to estimate prediction uncertainty based on:
   - Historical forecast errors
   - Model disagreement (ensemble dispersion)
   - Data quality scores